[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/notebooks/RO/capitol5_notebook_seminar.ipynb)

# Modelarea piețelor financiare — Seminarul 5: Volatilitate condiționată, modele Generalised AutoRegressive Conditional Heteroskedasticity (GARCH)

*Notebook de seminar. Daniel Traian Pele, Antoaneta Amza, Academia de Studii Economice din București.*

- Problemele **[Rezolvat]** conțin rezolvarea completă; problemele **[Propus]** le rezolvați dumneavoastră.
- Seminarul are loc înaintea cursului 5; slide-urile încep cu o scurtă secțiune introductivă (AutoRegressive Conditional Heteroskedasticity (ARCH) și GARCH, estimare și diagnostic, asimetrie și modele imbricate).
- **La seminar**: A1, B1, B2, B3, C1. **După curs**: A2, B4 și B5; celelalte secțiuni sînt extensii opționale.
- Partea A: derivări (A1–A6); Partea B: calcul și inferență (B1–B8); Partea C: întrebări deschise și critica unui răspuns Artificial Intelligence (AI) (C1 critica unui răspuns AI, C2 transmiterea șocurilor din Statele Unite ale Americii (SUA) către Bucharest Exchange Trading (index) (BET), C3 GARCH bazat pe cvasi-scor).

> Rezolvările se discută la seminar.

In [1]:
# the arch package (Kevin Sheppard) is not preinstalled on Colab
try:
    import arch
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch'], check=True)

Matplotlib is building the font cache; this may take a moment.


In [2]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats, optimize
from scipy.signal import lfilter
from scipy.special import gammaln
from arch import arch_model
from arch.univariate import ARCHInMean, ConstantMean, GARCH, FIGARCH, StudentsT, ZeroMean
import warnings
warnings.filterwarnings('ignore')

## Stilul graficelor și folderul rezultatelor

In [3]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
# font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 11.5
plt.rcParams['ytick.labelsize'] = 11.5
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.4
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 11

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
Teal     = '#17A2B8'
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'btc': Amber, 'eurron': Forest, 'gold': Purple}

SEED = 42
TABLE_DIR = '.'
NUM = {}


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def acf(x, nlags):
    x = np.asarray(x) - np.mean(x)
    d = np.sum(x ** 2)
    return np.array([np.sum(x[k:] * x[:-k]) / d for k in range(1, nlags + 1)])


import tempfile
# rezultatele merg în OUT_DIR (Google Drive) dacă SAVE_TO_DRIVE = True, altfel într-un folder temporar
OUT_PATH = OUT_DIR if globals().get('SAVE_TO_DRIVE') else tempfile.gettempdir()
TABLE_DIR = OUT_PATH


def save_fig(name):
    """Salvează figura ca PDF și PNG în OUT_PATH, apoi o afișează."""
    plt.savefig(os.path.join(OUT_PATH, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(OUT_PATH, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()

## Date

- Închideri zilnice pînă la 18 septembrie 2026; randamente în %: $r_t = 100(\ln P_t - \ln P_{t-1})$.
- Indici bursieri și aur: doar zilele lucrătoare, fără zilele cu închidere neschimbată (sărbători completate); aurul este anualizat cu frecvența sa reală.
- Bitcoin se tranzacționează în toate cele 7 zile ale săptămînii; euro (EUR)/Romanian leu (RON) este cursul oficial de referință Banca Națională a României (BNR).
- Fiecare serie este analizată pe propriul calendar.

In [4]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# name -> (symbol, label, group, start date)
MARKETS = {
    'sp500':  ('GSPC.INDX',      'S&P 500',                 'Equity', '2000-01-01'),
    'bet':    ('BET',            'BET',                     'Equity', '2000-01-01'),
    'bettr':  ('BETTR.INDX',     'BET-TR',                  'Equity', '2014-09-23'),
    'btc':    ('BTC-USD.CC',     'Bitcoin',                 'Crypto', '2014-09-17'),
    'eurron': ('REF:EUR',        'EUR/RON (BNR reference)', 'FX',     '2005-07-01'),
    'gold':   ('XAUUSD.FOREX',   'Gold (XAU/USD)',          'Commodity', '2000-01-01'),
}
LABELS = {k: v[1] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Daily price table of one symbol (local copy of the course data, or GitHub)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official RON reference rate (annual BNR XML archives)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END):
    """Daily closing price, cleaned with the chapter conventions."""
    symbol, _, group, start0 = MARKETS[name]
    start = start or start0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    s = read_market(symbol)['close'].loc[start:end]
    s = s[s > 0].dropna()
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    return s.rename(name)


def pct_returns(name, start=None, end=END):
    """Daily log returns in percent, on the series' own calendar."""
    return (100 * np.log(load_close(name, start, end)).diff().dropna()).rename(name)


def periods_per_year(r):
    """Average number of observations per calendar year (actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def load_vix(start='2000-01-01', end=END):
    """The VIX index (30-day implied volatility of the S&P 500, annualised percent)."""
    s = read_market('VIX.INDX')['close'].loc[start:end]
    return s[s.index.dayofweek < 5].rename('vix')


rets = {k: pct_returns(k) for k in MARKETS}
PPY = {k: periods_per_year(rets[k]) for k in MARKETS}
ORDER = list(MARKETS)
MAIN = ['sp500', 'bet', 'btc', 'eurron', 'gold']
for k in ORDER:
    print(f'{LABELS[k]:26s}', rets[k].index[0].date(), '->', rets[k].index[-1].date(), len(rets[k]), 'obs',
          round(PPY[k], 1), 'obs/year')

S&P 500                    2000-01-04 -> 2026-09-18 6714 obs 251.4 obs/year
BET                        2000-01-06 -> 2026-09-18 6670 obs 249.8 obs/year
BET-TR                     2014-09-24 -> 2026-09-18 2992 obs 249.7 obs/year
Bitcoin                    2014-09-18 -> 2026-09-18 4384 obs 365.3 obs/year
EUR/RON (BNR reference)    2005-07-05 -> 2026-09-18 5352 obs 252.4 obs/year
Gold (XAU/USD)             2000-01-04 -> 2026-09-18 6922 obs 259.2 obs/year


## Instrumente GARCH

- `fit_garch`: GARCH, Glosten–Jagannathan–Runkle GARCH (GJR-GARCH) sau Exponential GARCH (EGARCH) cu inovații Normale, Student-t, t asimetrice sau Generalised Error Distribution (GED) (pachetul `arch`); seriile cu volatilitate mică sînt rescalate intern, iar parametrii sînt convertiți înapoi.
- `half_life`, `news_impact`, `sign_bias_test`, `ljung_box`, `arch_lm`, `ewma_variance`, `qlike`, `mz_regression`, `dm_test`.

In [5]:
VOL_SPEC = {'GARCH': dict(vol='GARCH', p=1, o=0, q=1),
            'GJR': dict(vol='GARCH', p=1, o=1, q=1),
            'EGARCH': dict(vol='EGARCH', p=1, o=1, q=1)}
DISTS = {'normal': 'Normal', 't': 'Student-t', 'skewt': 'Skewed-t', 'ged': 'GED'}


class Fit:
    """Result of one fit, in the units of the original series (returns in %)."""

    def __init__(self, res, c, r, vol):
        self.res, self.c, self.r, self.vol = res, c, r, vol
        p = res.params.copy()
        p['mu'] = p['mu'] / c
        if vol == 'EGARCH':
            p['omega'] = p['omega'] - (1 - p['beta[1]']) * np.log(c ** 2)
        else:
            p['omega'] = p['omega'] / c ** 2
        self.params = p
        # robust (Bollerslev-Wooldridge) covariance mapped to the original units with the Jacobian
        # of the transformation: mu/c, omega/c^2 (GARCH, GJR) or omega - (1 - beta) ln c^2 (EGARCH)
        names = list(res.params.index)
        J = np.eye(len(names))
        J[names.index('mu'), names.index('mu')] = 1 / c
        if vol == 'EGARCH':
            J[names.index('omega'), names.index('beta[1]')] = np.log(c ** 2)
        else:
            J[names.index('omega'), names.index('omega')] = 1 / c ** 2
        self.cov = pd.DataFrame(J @ res.param_cov.values @ J.T, index=names, columns=names)
        self.se = pd.Series(np.sqrt(np.diag(self.cov.values)), index=names)
        self.tvalues = p[names] / self.se
        self.pvalues = pd.Series(2 * stats.norm.sf(np.abs(self.tvalues.values)), index=names)
        n = res.nobs
        self.nobs = n
        self.loglik = res.loglikelihood + n * np.log(c)
        k = len(p)
        self.aic = -2 * self.loglik + 2 * k
        self.bic = -2 * self.loglik + k * np.log(n)
        self.sigma = res.conditional_volatility / c
        self.z = res.std_resid
        self.eps = res.resid / c                   # unstandardised residuals, in the units of the original series
        self.converged = res.convergence_flag == 0

    @property
    def persistence(self):
        p = self.params
        if self.vol == 'EGARCH':
            return p['beta[1]']
        g = p.get('gamma[1]', 0.0)
        return p['alpha[1]'] + p['beta[1]'] + g * self.neg_share()

    def neg_share(self):
        """E[z^2 1(z < 0)] for the innovation distribution (0.5 if symmetric); enters the GJR persistence."""
        d = self.res.model.distribution
        if d.name.startswith('Standardized Skew'):
            from scipy import integrate
            par = self.res.params[d.parameter_names()].values
            pdf = lambda x: float(np.exp(d.loglikelihood(par, np.array([x]), np.array([1.0]), individual=True))[0])
            return float(integrate.quad(lambda x: x ** 2 * pdf(x), -np.inf, 0, limit=200)[0])
        return 0.5

    def uncond_var(self):
        p = self.params
        if self.vol == 'EGARCH':
            return np.nan
        return p['omega'] / (1 - self.persistence)

    def forecast_var(self, horizon):
        """Daily variance forecasts for t+1, ..., t+h, in %^2."""
        f = self.res.forecast(horizon=horizon, reindex=False, method='analytic' if self.vol != 'EGARCH' else 'simulation',
                              simulations=2000)
        return f.variance.iloc[-1].values / self.c ** 2


def scale_for(r):
    """Rescaling factor (a power of 10) that makes the standard deviation of order 1."""
    s = r.std()
    return 10.0 if s < 0.5 else 1.0


def fit_garch(r, vol='GARCH', dist='t', mean='Constant', last_obs=None, c=None):
    c = scale_for(r) if c is None else c
    am = arch_model(c * r, mean=mean, dist=dist, **VOL_SPEC[vol])
    res = am.fit(disp='off', last_obs=last_obs, options={'maxiter': 1000})
    return Fit(res, c, r, vol)


def half_life(persistence):
    """Number of days after which half of a variance shock has gone."""
    return np.log(0.5) / np.log(persistence) if 0 < persistence < 1 else np.inf


def news_impact(fit, eps, sigma2_bar=None):
    """sigma^2_t as a function of the shock eps_{t-1}, with sigma^2_{t-1} fixed at the sample variance."""
    p = fit.params
    s2 = fit.r.var() if sigma2_bar is None else sigma2_bar
    if fit.vol == 'EGARCH':
        z = eps / np.sqrt(s2)
        return np.exp(p['omega'] + p['alpha[1]'] * (np.abs(z) - np.sqrt(2 / np.pi)) + p['gamma[1]'] * z
                      + p['beta[1]'] * np.log(s2))
    g = p.get('gamma[1]', 0.0)
    return p['omega'] + (p['alpha[1]'] + g * (eps < 0)) * eps ** 2 + p['beta[1]'] * s2


def sign_bias_test(z, eps):
    """Engle-Ng (1993, eq. 18): z_t^2 on S-_{t-1}, S-_{t-1} eps_{t-1}, S+_{t-1} eps_{t-1}, with eps_{t-1} the
    UNSTANDARDISED residual and z_t the standardised residual; t statistics and the joint test T R^2 ~ chi2(3)."""
    d = pd.concat([pd.Series(z), pd.Series(eps)], axis=1, keys=['z', 'e']).dropna()
    z, e = d['z'].values, d['e'].values
    y = z[1:] ** 2
    el = e[:-1]
    sneg = (el < 0).astype(float)
    X = np.column_stack([np.ones_like(el), sneg, sneg * el, (1 - sneg) * el])
    ols = sm.OLS(y, X).fit()
    joint = len(y) * ols.rsquared
    return {'sign_t': ols.tvalues[1], 'neg_size_t': ols.tvalues[2], 'pos_size_t': ols.tvalues[3],
            'joint': joint, 'joint_p': 1 - stats.chi2.cdf(joint, 3)}


def ljung_box(x, lags=10):
    from statsmodels.stats.diagnostic import acorr_ljungbox
    lb = acorr_ljungbox(pd.Series(np.asarray(x)).dropna(), lags=[lags])
    return float(lb['lb_stat'].iloc[0]), float(lb['lb_pvalue'].iloc[0])


def arch_lm(x, lags=5):
    from statsmodels.stats.diagnostic import het_arch
    lm, lmp, _, _ = het_arch(pd.Series(np.asarray(x)).dropna(), nlags=lags)
    return float(lm), float(lmp)


def ewma_variance(r, lam=0.94, init=250):
    """EWMA forecast for day t+1 (aligned at t+1): s2_{t+1} = lam s2_t + (1 - lam) r_t^2."""
    x = r.values
    s2 = np.empty(len(x) + 1)
    s2[0] = np.mean(x[:init] ** 2)
    for t in range(len(x)):
        s2[t + 1] = lam * s2[t] + (1 - lam) * x[t] ** 2
    return pd.Series(s2[1:-1], index=r.index[1:])


def qlike(proxy, h):
    """QLIKE loss (Patton, 2011), the form used for comparisons: proxy/h + ln h."""
    return proxy / h + np.log(h)


def mz_regression(proxy, h, lags=5):
    """Mincer-Zarnowitz: proxy = a + b h + u, HAC errors; Wald test of a = 0, b = 1."""
    X = sm.add_constant(np.asarray(h))
    ols = sm.OLS(np.asarray(proxy), X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    w = ols.wald_test((np.eye(2), np.array([0.0, 1.0])), scalar=True)
    return {'a': ols.params[0], 'b': ols.params[1], 'a_se': ols.bse[0], 'b_se': ols.bse[1],
            'r2': ols.rsquared, 'wald': float(w.statistic), 'wald_p': float(w.pvalue)}


def dm_test(loss_a, loss_b, lags=5):
    """Diebold-Mariano: d = L_a - L_b; HAC t statistic for the mean of d (negative: A better)."""
    d = np.asarray(loss_a) - np.asarray(loss_b)
    ols = sm.OLS(d, np.ones_like(d)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    t = float(ols.tvalues[0])
    return {'mean_diff': float(d.mean()), 't': t, 'p': 2 * (1 - stats.norm.cdf(abs(t)))}


LBL = {'T': 'Forecast origin', 'rT': 'Return on the last day r_T (%)', 'eps': 'Shock eps_T = r_T - mu', 's2T': 'Variance sigma_T^2', 's2n': "Tomorrow's variance sigma_{T+1}^2", 'uv': 'Long-run variance', 'h10': 'Variance forecast for day T+10', 'sum10': '10-day variance (sum of the forecasts)', 'vol10_pct': '10-day volatility (%)', 'cond': 'Fourth-moment condition (alpha+beta)^2 + 2 alpha^2', 'K': 'Kurtosis K', 'corr': 'Correlation of GARCH-t volatility and the VIX', 'mean_vix': 'Mean VIX (%)', 'mean_garch': 'Mean GARCH-t volatility (%)', 'max_garch': 'Highest GARCH-t volatility (%)', 'max_garch_date': 'Date of the highest GARCH-t volatility', 'min_garch': 'Lowest GARCH-t volatility (%)', 'min_garch_date': 'Date of the lowest GARCH-t volatility', 'share_vix_above': 'Share of days with the VIX above GARCH-t', 'garch_max': 'Peak GARCH-t volatility, 2020 (%)', 'gjr_max': 'Peak GJR-t volatility, 2020 (%)', 'vix_max': 'Peak VIX, 2020', 'vix_max_date': 'Date of the VIX peak', 'garch_max_date': 'Date of the GARCH-t peak', 'rmin': 'Lowest return, 2020 (%)', 'rmin_date': 'Date of the lowest return', 'vix_0221': 'VIX on 21 Feb 2020', 'vix_0224': 'VIX on 24 Feb 2020', 'garch_0224': 'GARCH-t volatility on 24 Feb 2020 (%)', 'garch_0225': 'GARCH-t volatility on 25 Feb 2020 (%)', 'r_0224': 'Return on 24 Feb 2020 (%)', 'z_kurt': 'Excess kurtosis of z', 'z_skew': 'Skewness of z', 'jb': 'Jarque-Bera statistic', 'z_min': 'Smallest z', 'z_min_date': 'Date of the smallest z', 'nu_t': 'Student-t degrees of freedom nu (GARCH-t)', 'r_kurt': 'Excess kurtosis of the returns', 'n_below4': 'Residuals below -4', 'exp_below4_normal': 'Expected below -4, Normal', 'exp_below4_t': 'Expected below -4, Student-t', 'skewt_eta': 'Skewed-t degrees of freedom nu (GJR)', 'skewt_lambda': 'Skewed-t asymmetry lambda (GJR)', 'r_1': 'ACF of r_t, lag 1', 'r2_1': 'ACF of r_t^2, lag 1', 'r2_10': 'ACF of r_t^2, lag 10', 'r2_50': 'ACF of r_t^2, lag 50', 'band': '95% band', 'n_r2_sig': 'Significant lags of r_t^2', 'n_r_sig': 'Significant lags of r_t', 'kurt_garch': 'Excess kurtosis, simulated GARCH', 'acf1_x2': 'ACF of squared returns, lag 1', 'kurt_theory': 'Theoretical excess kurtosis', 'uvar': 'Unconditional variance', 's2bar': 'Long-run variance', 'gjr_m3': 'GJR variance after a -3% shock', 'gjr_p3': 'GJR variance after a +3% shock', 'eg_m3': 'EGARCH variance after a -3% shock', 'eg_p3': 'EGARCH variance after a +3% shock', 'g_pm3': 'GARCH variance after a shock of +/-3%', 'emp_1': 'ACF of |r_t|, lag 1, data', 'emp_100': 'ACF of |r_t|, lag 100, data', 'emp_250': 'ACF of |r_t|, lag 250, data', 'garch_1': 'ACF of |r_t|, lag 1, GARCH', 'garch_100': 'ACF of |r_t|, lag 100, GARCH', 'garch_250': 'ACF of |r_t|, lag 250, GARCH', 'fig_1': 'ACF of |r_t|, lag 1, FIGARCH', 'fig_100': 'ACF of |r_t|, lag 100, FIGARCH', 'fig_250': 'ACF of |r_t|, lag 250, FIGARCH', 'd': 'FIGARCH memory parameter d', 'd_se': 's.e. of d', 'fig_bic': 'BIC, FIGARCH', 'garch_bic': 'BIC, GARCH', 'kappa_z': 'Kurtosis of z', 'theory_ratio': 'Theoretical s.e. ratio sqrt((kappa_z - 1)/2)', 'ratio_alpha': 'Observed robust/classical ratio, alpha', 'ratio_beta': 'Observed robust/classical ratio, beta', 'kappa_half': 'Kurtosis of z, first half of the sample', 'alpha': 'alpha', 'beta': 'beta', 'gamma': 'gamma', 'lr': 'LR statistic', 'p_mix': 'p-value, mixture 1/2 chi2(0) + 1/2 chi2(1)', 'nu': 'nu', 'crit_mix': '5% critical value, mixture', 'crit_chi2': '5% critical value, chi2(1)', 'gjr_alpha': 'alpha of GJR-t, S&P 500', 'pers': 'Persistence p = alpha + beta', 'se_p': 's.e. of p', 's2': 'Long-run variance', 'se_s2': 's.e. of the long-run variance', 'vol': 'Long-run volatility (% p.a.)', 'se_vol': 's.e. of the long-run volatility', 'hl': 'Half-life (days)', 'dh': 'dh/dp', 'se_hl': 's.e. of the half-life (delta method)', 'p_lo': '95% interval for p, lower end', 'p_hi': '95% interval for p, upper end', 'hl_lo': 'Half-life at the lower end of p', 'rho1': 'rho_1(eps^2), model', 'rho10': 'rho_10(eps^2), model', 'rho1_sample': 'rho_1(eps^2), sample', 'rho10_sample': 'rho_10(eps^2), sample', 'cond3': 'Condition, estimates rounded to 3 decimals', 'K3': 'Kurtosis K, estimates rounded to 3 decimals', 'rho13': 'rho_1, rounded estimates', 'M': 'Number of lags M', 'q_lb': 'Ljung-Box Q on z^2 (naive)', 'p_lb': 'p-value, naive', 'q_lm': 'Li-Mak corrected statistic', 'p_lm': 'p-value, Li-Mak', 'r1': 'First autocorrelation of z^2', 'c11': 'C[1,1]', 'c55': 'C[5,5]', 'cMM': 'C[M,M]', 'trace_loss': 'M - trace(C)', 'eta': 'Skewed-t degrees of freedom nu', 'lam': 'Skewed-t asymmetry lambda', 'm_neg': 'E[z^2 1(z<0)]', 'p_neg': 'P(z<0)', 'pers_sym': 'Persistence with 1/2', 'pers_skew': 'Persistence with E[z^2 1(z<0)]', 'll_garch': 'Log-likelihood, GARCH', 'll_gjr': 'Log-likelihood, GJR', 'aic_garch': 'AIC, GARCH', 'aic_gjr': 'AIC, GJR', 'bic_garch': 'BIC, GARCH', 'bic_gjr': 'BIC, GJR', 'lr_p': 'LR p-value, chi2(1)', 'gamma_t': 'Robust t of gamma', 'B': 'Bootstrap paths B', 'est': 'Estimated persistence p', 'pers_lo': '2.5% percentile of p*', 'pers_hi': '97.5% percentile of p*', 'pers_sd': 'Standard deviation of p*', 'pers_mean': 'Mean of p*', 'hl_med': 'Median half-life (days)', 'hl_hi': '97.5% percentile of the half-life', 'share_inf': 'Share of draws with an infinite half-life', 'share_ge_0999': 'Share of draws with p* >= 0.999', 'se_robust_sum': 'Robust s.e. of alpha + beta', 'alpha0': 'alpha under H0 (IGARCH-t)', 'omega0': 'omega under H0', 'nu0': 'nu under H0', 'p_chi2': 'p-value, naive chi2(1)', 'p_boot_pers': 'Bootstrap p-value of alpha + beta', 'p_boot_lr': 'Bootstrap p-value of LR', 'share_bound': 'Share of re-estimates with alpha + beta >= 0.999', 'lr_q95': '95% quantile of LR*', 'pers_q05': '5% quantile of p* under H0'}


def show(title, d, skip=()):
    """Print a dictionary of results as labelled lines."""
    print(title)
    for k, v in d.items():
        if k in skip or isinstance(v, (dict, list)):
            continue
        s = f'{v:.4f}' if isinstance(v, float) else str(v)
        print(f'  {LBL.get(k, k)}: {s}')


## Funcțiile seminarului

- Grila de modele, tabelul de asimetrie și prognozele în afara eșantionului (din curs), și funcțiile folosite în Părțile A–C.

In [6]:
MODELS_OOS = [('GARCH', 'normal', 'GARCH-N'), ('GARCH', 't', 'GARCH-t'), ('GJR', 't', 'GJR-t'), ('EGARCH', 't', 'EGARCH-t')]
OUT = {}
DRAWS = {}   # extragerile bootstrap (B4, B8)


def model_grid():
    rows, fits = [], {}
    for k in ORDER:
        r = rets[k]
        for vol in VOL_SPEC:
            for dist in DISTS:
                f = fit_garch(r, vol, dist)
                fits[(k, vol, dist)] = f
                p = f.params
                rows.append(dict(market=k, vol=vol, dist=dist, loglik=f.loglik, aic=f.aic, bic=f.bic,
                                 k=len(p), converged=f.converged, persistence=f.persistence,
                                 **{f'p_{n}': v for n, v in p.items()}))
    g = pd.DataFrame(rows)
    g['dbic'] = g['bic'] - g.groupby('market')['bic'].transform('min')
    g.to_csv(os.path.join(TABLE_DIR, 'ch5_model_grid.csv'), index=False)
    return g, fits


def garch_table(fits):
    """GARCH(1,1)-t on each series: parameters, robust and classical errors, persistence, half-life."""
    rows = []
    for k in ORDER:
        f = fits[(k, 'GARCH', 't')]
        p, se = f.params, f.se
        classic = f.res.model.fit(disp='off', cov_type='classic', starting_values=f.res.params.values).std_err
        pers = p['alpha[1]'] + p['beta[1]']
        uv = f.uncond_var()
        rows.append(dict(market=k, mu=p['mu'], omega=p['omega'], alpha=p['alpha[1]'], beta=p['beta[1]'], nu=p['nu'],
                         se_alpha=se['alpha[1]'], se_beta=se['beta[1]'], se_nu=se['nu'],
                         se_alpha_classic=classic['alpha[1]'], se_beta_classic=classic['beta[1]'],
                         persistence=pers, half_life=half_life(pers),
                         uncond_ann_vol=np.sqrt(uv * PPY[k]) if pers < 0.999 else np.nan,
                         last_ann_vol=f.sigma.iloc[-1] * np.sqrt(PPY[k]), loglik=f.loglik, N=f.nobs, scale=f.c))
    t = pd.DataFrame(rows).set_index('market')
    t.to_csv(os.path.join(TABLE_DIR, 'ch5_garch_table.csv'))
    return t


def asym_table(fits):
    rows = []
    for k in ORDER:
        g, j, e = fits[(k, 'GARCH', 't')], fits[(k, 'GJR', 't')], fits[(k, 'EGARCH', 't')]
        lr = 2 * (j.loglik - g.loglik)
        rows.append(dict(market=k, gjr_alpha=j.params['alpha[1]'], gjr_gamma=j.params['gamma[1]'],
                         gjr_gamma_t=j.tvalues['gamma[1]'], gjr_beta=j.params['beta[1]'],
                         gjr_persistence=j.persistence, lr=lr, lr_p=1 - stats.chi2.cdf(lr, 1),
                         eg_alpha=e.params['alpha[1]'], eg_gamma=e.params['gamma[1]'], eg_gamma_t=e.tvalues['gamma[1]'],
                         eg_beta=e.params['beta[1]'], bic_garch=g.bic, bic_gjr=j.bic, bic_egarch=e.bic))
    t = pd.DataFrame(rows).set_index('market')
    t.to_csv(os.path.join(TABLE_DIR, 'ch5_asym_table.csv'))
    return t


def oos_forecasts(k='sp500', first_year=2016, H=22):
    r = rets[k]
    idx = r.index
    last_year = idx[-1].year
    out1 = {lab: pd.Series(np.nan, index=idx) for *_, lab in MODELS_OOS}
    outH = {lab: pd.Series(np.nan, index=idx) for *_, lab in MODELS_OOS if lab != 'EGARCH-t'}
    for y in range(first_year, last_year + 1):
        start = idx[idx < f'{y}-01-01'][-1]
        end = idx[idx <= f'{y}-12-31'][-1]
        orig = idx[(idx >= start) & (idx < end)]
        for vol, dist, lab in MODELS_OOS:
            f = fit_garch(r, vol, dist, last_obs=idx[idx >= f'{y}-01-01'][0])
            hh = 1 if vol == 'EGARCH' else H
            fc = f.res.forecast(horizon=hh, start=start, reindex=False).variance / f.c ** 2
            fc = fc.loc[orig]
            pos = idx.get_indexer(fc.index)
            out1[lab].iloc[pos + 1] = fc.iloc[:, 0].values
            if hh == H and lab in outH:
                outH[lab].loc[fc.index] = fc.sum(axis=1).values      # cumulative variance over t+1..t+H, at origin t
    ew = ewma_variance(r)
    hist = (r ** 2).rolling(20).mean().shift(1)
    out1['EWMA'] = ew.reindex(idx)
    out1['Hist-20'] = hist
    outH['EWMA'] = (ew.shift(-1) * H).reindex(idx)       # the EWMA forecast is flat across horizons
    f1 = pd.DataFrame(out1).loc[f'{first_year}-01-01':].dropna()
    fH = pd.DataFrame(outH).dropna()
    return f1, fH


def a1_recursion():
    om, al, be = 0.05, 0.10, 0.85
    eps = [-2.0, 0.5, 3.0, -1.0]
    s2 = [om / (1 - al - be)]
    for e in eps:
        s2.append(om + al * e ** 2 + be * s2[-1])
    uv = om / (1 - al - be)
    last = s2[-1]
    fc = [uv + (al + be) ** (h - 1) * (last - uv) for h in range(1, 11)]
    OUT['a1'] = {'omega': om, 'alpha': al, 'beta': be, 'eps': eps, 's2': s2, 'uv': uv, 'fc': fc,
                 'hl': half_life(al + be), 'sum5': float(np.sum(fc[:5])), 'sum10': float(np.sum(fc))}


def lr_tests():
    at = pd.read_csv(os.path.join(TABLE_DIR, 'ch5_asym_table.csv'), index_col=0)
    grid = pd.read_csv(os.path.join(TABLE_DIR, 'ch5_model_grid.csv'))
    res = {}
    for k in ['sp500', 'bet']:
        sub = grid[(grid.market == k) & (grid.dist == 't')].set_index('vol')
        res[k] = {'ll_garch': sub.loc['GARCH', 'loglik'], 'll_gjr': sub.loc['GJR', 'loglik'],
                  'aic_garch': sub.loc['GARCH', 'aic'], 'aic_gjr': sub.loc['GJR', 'aic'],
                  'bic_garch': sub.loc['GARCH', 'bic'], 'bic_gjr': sub.loc['GJR', 'bic'],
                  'lr': at.loc[k, 'lr'], 'lr_p': at.loc[k, 'lr_p'], 'gamma': at.loc[k, 'gjr_gamma'],
                  'gamma_t': at.loc[k, 'gjr_gamma_t'], 'alpha': at.loc[k, 'gjr_alpha'], 'beta': at.loc[k, 'gjr_beta']}
    OUT['lr'] = res


def forecast_comparison(k):
    r = rets[k]
    f1, _ = oos_forecasts(k, 2016, 1)
    proxy = (r ** 2).reindex(f1.index)
    base = qlike(proxy, f1['GARCH-t'])
    rows = {}
    for m in f1.columns:
        L = qlike(proxy, f1[m])
        dm = dm_test(L, base) if m != 'GARCH-t' else {'mean_diff': 0.0, 't': float('nan'), 'p': float('nan')}
        mz = mz_regression(proxy, f1[m])
        rows[m] = {'qlike': float(L.mean()), 'dm_diff': dm['mean_diff'], 'dm_t': dm['t'], 'dm_p': dm['p'],
                   'mz_a': mz['a'], 'mz_b': mz['b'], 'mz_b_se': mz['b_se'], 'mz_r2': mz['r2'], 'mz_wald_p': mz['wald_p']}
    OUT[f'fc_{k}'] = {'n': len(f1), 'start': str(f1.index[0].date()), 'models': rows}


def tgarch_negll(theta, r, igarch=False):
    """Minus the log-likelihood of GARCH(1,1)-t with constant mean mu.
    Parametrisation: p = alpha + beta in [0, 1], s = alpha / p in [0, 1]; IGARCH: p = 1.
    Start as in arch: sigma^2_0 = eps^2_0 = EWMA-weighted mean (0.94) of the first 75 squares."""
    if igarch:
        mu, om, s, nu = theta
        p = 1.0
    else:
        mu, om, p, s, nu = theta
    e = r - mu
    al, be = p * s, p * (1 - s)
    w = 0.94 ** np.arange(75)
    bc = np.sum(e[:75] ** 2 * w) / w.sum()
    drive = om + al * np.concatenate(([bc], e[:-1] ** 2))
    s2 = lfilter([1.0], [1.0, -be], drive, zi=[be * bc])[0]
    z2 = e ** 2 / s2
    ll = (gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(np.pi * (nu - 2)) - 0.5 * np.log(s2)
          - (nu + 1) / 2 * np.log1p(z2 / (nu - 2)))
    return -ll.sum()


def fit_tgarch(r, igarch=False, start=None):
    """GARCH(1,1)-t (or IGARCH(1,1)-t, alpha + beta = 1) by our own likelihood, with the restriction alpha + beta <= 1.
    The unrestricted model is also started from the IGARCH solution: l_GARCH >= l_IGARCH, so LR >= 0."""
    r = np.asarray(r, dtype=float)
    m, v = np.mean(r), np.var(r)
    opts = {'ftol': 1e-12, 'gtol': 1e-7, 'maxiter': 3000}
    if igarch:
        x0 = [m, 0.01 * v, 0.1, 6.0] if start is None else start
        bnd = [(None, None), (1e-8 * v, 10 * v), (1e-4, 1.0), (2.05, 200.0)]
        o = optimize.minimize(tgarch_negll, x0, args=(r, True), method='L-BFGS-B', bounds=bnd, options=opts)
        mu, om, s, nu = o.x
        return {'mu': mu, 'omega': om, 'alpha': s, 'beta': 1 - s, 'nu': nu, 'pers': 1.0, 'loglik': -o.fun}
    g = fit_tgarch(r, igarch=True)
    bnd = [(None, None), (1e-8 * v, 10 * v), (0.0, 1.0), (1e-4, 1.0), (2.05, 200.0)]
    starts = [[m, 0.01 * v, 0.98, 0.1, 6.0], [g['mu'], g['omega'], 0.999, g['alpha'], g['nu']]]
    if start is not None:
        starts.append(start)
    best = min((optimize.minimize(tgarch_negll, x0, args=(r, False), method='L-BFGS-B', bounds=bnd, options=opts)
                for x0 in starts), key=lambda o: o.fun)
    mu, om, p, s, nu = best.x
    return {'mu': mu, 'omega': om, 'alpha': p * s, 'beta': p * (1 - s), 'nu': nu, 'pers': p,
            'loglik': max(-best.fun, g['loglik']), 'loglik_igarch': g['loglik']}


def sim_tgarch(par, n, B, rng, s2_start, burn=500):
    """B GARCH(1,1)-t paths at once (vectorised over columns); also works for alpha + beta = 1."""
    nu = par['nu']
    z = rng.standard_t(nu, size=(n + burn, B)) * np.sqrt((nu - 2) / nu)
    s2 = np.full(B, s2_start)
    out = np.empty((n + burn, B))
    for t in range(n + burn):
        e = np.sqrt(s2) * z[t]
        out[t] = e
        s2 = par['omega'] + par['alpha'] * e ** 2 + par['beta'] * s2
    return par['mu'] + out[burn:]


def param_bootstrap(k, B=999):
    """Parametric bootstrap: B paths from the fitted GARCH(1,1)-t, re-estimated; percentile intervals."""
    from joblib import Parallel, delayed
    r = rets[k]
    f = fit_garch(r, 'GARCH', 't')
    p = f.params
    par = {'mu': p['mu'], 'omega': p['omega'], 'alpha': p['alpha[1]'], 'beta': p['beta[1]'], 'nu': p['nu']}
    rng = np.random.default_rng(SEED)
    sims = sim_tgarch(par, len(r), B, rng, r.var())
    st = [par['mu'], par['omega'], par['alpha'] + par['beta'], par['alpha'] / (par['alpha'] + par['beta']), par['nu']]
    fits = Parallel(n_jobs=-1)(delayed(fit_tgarch)(sims[:, b], False, st) for b in range(B))
    pers = np.array([x['pers'] for x in fits])
    hls = np.array([half_life(x) for x in pers])
    hq = np.sort(hls)                                          # np.inf kept: persistence at the bound 1
    DRAWS[f'boot_pers_{k}'] = pers
    est = p['alpha[1]'] + p['beta[1]']
    OUT[f'boot_{k}'] = {'B': B, 'est': float(est), 'hl': float(half_life(est)),
                        'pers_lo': float(np.quantile(pers, 0.025)), 'pers_hi': float(np.quantile(pers, 0.975)),
                        'pers_sd': float(pers.std()), 'pers_mean': float(pers.mean()),
                        'hl_lo': float(np.quantile(hq, 0.025)), 'hl_med': float(np.median(hq)),
                        # h(p) is increasing: the upper end is h(97.5% quantile of p), infinite if p = 1
                        'hl_hi': float(half_life(np.quantile(pers, 0.975))), 'share_inf': float((pers >= 1 - 1e-6).mean()),
                        'share_ge_0999': float((pers >= 0.999).mean()),
                        'se_robust_sum': float(np.sqrt(f.res.param_cov.loc[['alpha[1]', 'beta[1]'], ['alpha[1]', 'beta[1]']].values.sum()))}
    return pers, hls


def _null_rep(y):
    ub = fit_tgarch(y)
    return ub['pers'], 2 * (ub['loglik'] - ub['loglik_igarch'])


def igarch_test(k, B=999):
    """Test of H0: alpha + beta = 1 (IGARCH) by parametric bootstrap UNDER H0 (Andrews, 2000: a percentile interval
    truncated at 1 is not a valid test). Statistics: the estimated persistence and LR = 2(l_GARCH - l_IGARCH)."""
    from joblib import Parallel, delayed
    r = rets[k]
    u = fit_tgarch(r)
    g0 = fit_tgarch(r, igarch=True)
    lr = 2 * (u['loglik'] - g0['loglik'])
    rng = np.random.default_rng(SEED + 1)
    sims = sim_tgarch(g0, len(r), B, rng, r.var())
    res = Parallel(n_jobs=-1)(delayed(_null_rep)(sims[:, b]) for b in range(B))
    pb, lrb = np.array([x[0] for x in res]), np.array([x[1] for x in res])
    DRAWS[f'null_pers_{k}'], DRAWS[f'null_lr_{k}'] = pb, lrb
    OUT[f'igarch_{k}'] = {'B': B, 'pers': u['pers'], 'alpha0': g0['alpha'], 'omega0': g0['omega'], 'nu0': g0['nu'],
                          'lr': lr, 'p_chi2': float(stats.chi2.sf(lr, 1)), 'p_mix': float(0.5 * stats.chi2.sf(lr, 1)),
                          'p_boot_pers': float((1 + np.sum(pb <= u['pers'])) / (B + 1)),
                          'p_boot_lr': float((1 + np.sum(lrb >= lr)) / (B + 1)),
                          'share_bound': float(np.mean(pb >= 0.999)), 'lr_q95': float(np.quantile(lrb, 0.95)),
                          'pers_q05': float(np.quantile(pb, 0.05))}
    return pb


from scipy import integrate
from arch.bootstrap import MCS
INF = {}


def delta_method():
    """Delta method from the robust covariance of (omega, alpha, beta), GARCH(1,1)-t, S&P 500."""
    f = fit_garch(rets['sp500'], 'GARCH', 't')
    p = f.params
    V = f.res.param_cov.loc[['omega', 'alpha[1]', 'beta[1]'], ['omega', 'alpha[1]', 'beta[1]']].values / f.c ** 2
    om, a, b = p['omega'], p['alpha[1]'], p['beta[1]']
    pers = a + b
    se_p = float(np.sqrt(V[1:, 1:].sum()))
    s2 = om / (1 - pers)
    g = np.array([1 / (1 - pers), om / (1 - pers) ** 2, om / (1 - pers) ** 2])
    se_s2 = float(np.sqrt(g @ V @ g))
    vol = np.sqrt(PPY['sp500'] * s2)
    se_vol = PPY['sp500'] * se_s2 / (2 * vol)
    hl = half_life(pers)
    dh = -np.log(0.5) / (pers * np.log(pers) ** 2)
    lo, hi = pers - 1.96 * se_p, pers + 1.96 * se_p
    INF['delta'] = {'pers': float(pers), 'se_p': se_p, 's2': float(s2), 'se_s2': se_s2, 'vol': float(vol),
                    'se_vol': float(se_vol), 'hl': float(hl), 'dh': float(dh), 'se_hl': float(dh * se_p),
                    'p_lo': float(lo), 'p_hi': float(hi), 'hl_lo': float(half_life(lo)),
                    'grad_om': float(g[0]), 'grad_ab': float(g[1])}


def oos_rolling(k='sp500', first_year=2016, W=1000):
    """One-day forecasts with a ROLLING window of W days, refit every January (Giacomini-White setting)."""
    r = rets[k]
    idx = r.index
    out = {lab: pd.Series(np.nan, index=idx) for *_, lab in MODELS_OOS}
    for y in range(first_year, idx[-1].year + 1):
        start = idx[idx < f'{y}-01-01'][-1]
        end = idx[idx <= f'{y}-12-31'][-1]
        orig = idx[(idx >= start) & (idx < end)]
        pos0 = idx.get_loc(start)
        sub = r.iloc[pos0 - W + 1:]
        for vol, dist, lab in MODELS_OOS:
            f = fit_garch(sub, vol, dist, last_obs=idx[pos0 + 1])
            fc = f.res.forecast(horizon=1, start=start, reindex=False).variance / f.c ** 2
            fc = fc.loc[orig]
            pos = idx.get_indexer(fc.index)
            out[lab].iloc[pos + 1] = fc.iloc[:, 0].values
    out['EWMA'] = ewma_variance(r).reindex(idx)
    out['Hist-20'] = (r ** 2).rolling(20).mean().shift(1)
    return pd.DataFrame(out).loc[f'{first_year}-01-01':].dropna()


def gw_test(la, lb):
    """Giacomini-White (2006), conditional test, horizon 1: instruments x_t = (1, d_t); GW = n Zbar' Omega^{-1} Zbar ~ chi2(2)."""
    d = np.asarray(la) - np.asarray(lb)
    Z = np.column_stack([d[1:], d[:-1] * d[1:]])
    zb = Z.mean(axis=0)
    Om = Z.T @ Z / len(Z)
    stat = len(Z) * zb @ np.linalg.solve(Om, zb)
    return float(stat), float(stats.chi2.sf(stat, 2))


def gw_mcs(k='sp500'):
    r = rets[k]
    fr = oos_rolling(k)
    proxy = (r ** 2).reindex(fr.index)
    L = pd.DataFrame({m: qlike(proxy, fr[m]) for m in fr.columns})
    rows = {}
    for m in L.columns:
        if m == 'GARCH-t':
            continue
        dm = dm_test(L[m], L['GARCH-t'])
        gw, gp = gw_test(L[m], L['GARCH-t'])
        rows[m] = {'uncond_t': dm['t'], 'gw': gw, 'gw_p': gp, 'mean_diff': dm['mean_diff']}
    # MCS on the expanding-window forecasts (the table of the lecture and seminar)
    f1, _ = oos_forecasts(k, 2016, 1)
    px = (r ** 2).reindex(f1.index)
    Le = pd.DataFrame({m: qlike(px, f1[m]) for m in f1.columns})
    mcs = MCS(Le, size=0.10, reps=10000, block_size=10, method='R', bootstrap='stationary', seed=SEED)
    mcs.compute()
    pv = mcs.pvalues['Pvalue']
    INF[f'gw_{k}'] = {'n': len(fr), 'W': 1000, 'qlike_roll': L.mean().to_dict(), 'tests': rows,
                      'mcs_p': pv.to_dict(), 'mcs_in': list(mcs.included), 'n_mcs': len(Le)}


grid, FITS = model_grid()
gt = garch_table(FITS)
at = asym_table(FITS)

## Partea A — derivări

### A1. Prognoza dispersiei pe mai mulți pași: derivare și cifre [Rezolvat]

- Deduceți $\mathbb{E}_t\sigma_{t+h}^2 = \bar\sigma^2 + p^{h-1}(\sigma_{t+1}^2 - \bar\sigma^2)$ și $V_H = H\bar\sigma^2 + (\sigma_{t+1}^2 - \bar\sigma^2)(1-p^H)/(1-p)$, $p = \alpha+\beta$.
- $\omega = 0{,}05$, $\alpha = 0{,}10$, $\beta = 0{,}85$; șocurile $-2$; $0{,}5$; $3$; $-1$: traiectoria, prognozele, $V_5$ din sumă și din forma explicită, $5\sigma_5^2$ și timpul de înjumătățire.

In [7]:
# Rezolvare
a1_recursion()
a1 = OUT['a1']
p = a1['alpha'] + a1['beta']
print('sigma^2 path:', np.round(a1['s2'], 4))
print('forecasts h = 1..5:', np.round(a1['fc'][:5], 4))
vh = 5 * a1['uv'] + (a1['s2'][-1] - a1['uv']) * (1 - p ** 5) / (1 - p)
print('V_5: sum', round(a1['sum5'], 3), ' closed form', round(vh, 3), ' 5 sigma_5^2', round(5 * a1['s2'][-1], 3), ' half-life', round(a1['hl'], 1))

sigma^2 path: [1.    1.3   1.18  1.953 1.81 ]
forecasts h = 1..5: [1.81   1.7695 1.7311 1.6945 1.6598]
V_5: sum 8.665  closed form 8.665  5 sigma_5^2 9.05  half-life 13.5


### A2. Volatilitatea de termen lung, timpul de înjumătățire și erorile standard prin metoda delta [Rezolvat] — după curs

- S&P 500 GARCH(1,1)-t: $\sigma^2 = \omega/(1-\alpha-\beta)$, volatilitatea zilnică și anuală, timpul de înjumătățire; apoi erorile standard prin metoda delta din covarianța robustă a lui $(\hat\omega, \hat\alpha, \hat\beta)$.

In [8]:
# Rezolvare
p = gt.loc['sp500']
uv = p['omega'] / (1 - p['persistence'])
print('alpha+beta', round(p['persistence'], 4), ' long-run variance', round(uv, 3), ' daily vol', round(np.sqrt(uv), 3))
print('annual vol', round(np.sqrt(uv * PPY['sp500']), 1), ' sample', round(rets['sp500'].std() * np.sqrt(PPY['sp500']), 1))
print('half-life', round(half_life(p['persistence']), 1), 'days')
delta_method()
show('Delta method', INF['delta'], skip=('grad_om', 'grad_ab'))

alpha+beta 0.9947  long-run variance 2.972  daily vol 1.724
annual vol 27.3  sample 19.2
half-life 130.7 days
Delta method
  Persistence p = alpha + beta: 0.9947
  s.e. of p: 0.0047
  Long-run variance: 2.9717
  s.e. of the long-run variance: 2.3499
  Long-run volatility (% p.a.): 27.3336
  s.e. of the long-run volatility: 10.8073
  Half-life (days): 130.7335
  dh/dp: 24788.5443
  s.e. of the half-life (delta method): 115.9302
  95% interval for p, lower end: 0.9855
  95% interval for p, upper end: 1.0039
  Half-life at the lower end of p: 47.6066


### A3. Persistența Glosten–Jagannathan–Runkle (GJR) cu inovații asimetrice [Propus]

- Replicare: calcul/cod reproductibil, comparație cu [reperul pentru inovații asimetrice](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=sem5x1) și interpretare.

- **Întrebarea**: care este persistența GJR-GARCH cînd $z_t$ este asimetric?

- **Date**: S&P 500, GJR-GARCH(1,1) cu inovații t asimetrice [Hansen (1994)](https://doi.org/10.2307/2527081): $\hat\alpha = 0{,}00000$, $\hat\gamma = 0{,}21116$, $\hat\beta = 0{,}87965$, $\hat\nu = 7{,}70$, $\hat\lambda = -0{,}150$; densitatea standardizată $f$ a lui $z_t$ se obține prin exponențierea rezultatului `SkewStudent().loglikelihood` din `arch` (dispersie unitară, `individual=True`); $\lambda$: parametrul de asimetrie

- **Cerințe**

  1.  Luați media ambelor părți ale recursiei și arătați că persistența este $\alpha + \gamma\,m_- + \beta$, cu $m_- = \mathbb{E}[z_t^2 1(z_t < 0)]$.

  2.  Arătați că $m_- = 1/2$ pentru $z_t$ simetric și explicați de ce $m_-$ diferă în general de $P(z_t < 0)$.

  3.  Integrați numeric `quad(f, -inf, 0)` și `quad(lambda z: z**2*f(z), -inf, 0)`; verificați $\int f = 1$ și $\int z^2 f = 1$ pe toată axa.

  4.  Calculați persistența și timpul de înjumătățire cu $m_-$ și cu $1/2$.

- **Raportați**: $m_-$, $P(z<0)$, ambele persistențe și ambii timpi de înjumătățire; interpretare: ce se întîmplă dacă se folosește $\alpha + \gamma/2 + \beta$ cu inovații asimetrice?

- Model: [A1](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A1)

- Notebook: secțiunea [A3](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A3)

[Exercițiul A3 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A3)

In [9]:
# Codul vostru aici


### A4. Existența momentului de ordin patru [Propus]

- Replicare: calcul/cod reproductibil, comparație cu reperul din curs și interpretare.

- **Întrebările**

  - au modelele estimate pentru S&P 500 un kurtosis finit?

  - le poate valida kurtosis-ul de selecție?

- **Date**: GARCH with Normal innovations (GARCH-N): $\hat\alpha = 0{,}120604$, $\hat\beta = 0{,}859878$; GARCH with Student t innovations (GARCH-t): $\hat\alpha = 0{,}1226$, $\hat\beta = 0{,}8721$, $\hat\nu = 6{,}256$; kurtosis-ul de selecție al randamentelor 13,7 (total, nu exces)

- **Cerințe**

  1.  Cu $\kappa_z = \mathbb{E} z_t^4$ (kurtosis-ul lui $z_t$), arătați că $\mathbb{E}\varepsilon_t^4 = \kappa_z\mathbb{E}\sigma_t^4$ și $\mathbb{E}\sigma_t^4 < \infty$ dacă și numai dacă $\mathbb{E}(\alpha z_t^2 + \beta)^2 = (\alpha+\beta)^2 + (\kappa_z - 1)\alpha^2 < 1$.

  2.  Pentru GARCH-N ($\kappa_z = 3$) calculați condiția și kurtosis-ul total $K = 3[1-(\alpha+\beta)^2]/[1-(\alpha+\beta)^2-2\alpha^2]$, cu numitorul afișat; repetați cu estimările rotunjite la 3 zecimale.

  3.  Pentru GARCH-t calculați $\kappa_z = 3(\nu-2)/(\nu-4)$ și condiția.

- **Raportați**: un rînd pentru fiecare model: $\kappa_z$, condiția, finit sau nu, $K$; interpretare: este concordanța cu kurtosis-ul de selecție un test al modelului?

- Model: [A1](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A1), [A2](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A2)

- Notebook: secțiunea [A4](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A4)

[Exercițiul A4 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A4)

In [10]:
# Codul vostru aici


### A5. Dispersia robustă și cea clasică a Quasi-Maximum Likelihood Estimator (QMLE) Gaussian, ARCH(1) [Propus]

- Replicare: calcul/cod reproductibil, comparație cu reperul din curs și interpretare.

- **Întrebarea**: cît de mare este eroarea inversei hessianei ca estimator al dispersiei QMLE Gaussian cînd $z_t$ nu este Normal?

- **Date**: ARCH(1), $\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2$, $\omega$ cunoscut; $\ell_t = -\tfrac12(\ln\sigma_t^2 + \varepsilon_t^2/\sigma_t^2)$; $x_t = \varepsilon_{t-1}^2/\sigma_t^2$ este cunoscut la $t-1$ și independent de $z_t$; identități $\partial\sigma_t^2/\partial\alpha = \varepsilon_{t-1}^2$, $\varepsilon_t^2/\sigma_t^2 = z_t^2$

- **Cerințe**

  1.  Arătați că scorul este $s_t = \tfrac12(z_t^2 - 1)x_t$ și că $B = \mathbb{E} s_t^2 = \tfrac14(\kappa_z - 1)\mathbb{E} x_t^2$.

  2.  Arătați că $\partial^2\ell_t/\partial\alpha^2 = \tfrac12 x_t^2(1 - 2z_t^2)$ și $A = \tfrac12\mathbb{E} x_t^2$.

  3.  Arătați că dispersia asimptotică a lui $\sqrt{n}(\hat\alpha - \alpha)$ este $A^{-1}BA^{-1} = \tfrac{\kappa_z - 1}{2}A^{-1}$; dispersia în eșantion finit o împarte la $n$.

  4.  Calculați raportul e.s. $\sqrt{(\kappa_z - 1)/2}$ pentru Student-t cu $\nu = 6{,}26$ și pentru $\hat\kappa_z = 4{,}73$.

- **Raportați**: $s_t$, $A$, $B$, raportul, două cifre; interpretare: ce se întîmplă cu teoria în $\sqrt{n}$ cînd $\kappa_z = \infty$?

- Model: [A2](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A2)

- Notebook: secțiunea [A5](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A5)

[Exercițiul A5 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A5)

In [11]:
# Codul vostru aici


### A6. Autocorelația pătratelor șocurilor [Propus]

- Replicare: calcul/cod reproductibil, comparație cu reperul din curs și interpretare.

- **Întrebările**

  - este AutoCorrelation Function (ACF) a lui $r_t^2$ implicată de GARCH-N compatibilă cu datele?

  - cît de precisă este ACF de selecție?

- **Date**: forma AutoRegressive Moving Average (model) (ARMA)(1,1) a lui $\varepsilon_t^2$ ([[slide: Forma ARMA, Probability Integral Transform (PIT) și mulțimi de modele]{style="color: MainBlue"}](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=sem5x2)); ARMA(1,1) cu coeficientul autoregressive model (AR) $\phi$ și coeficientul Moving Average (MA) $\theta$: $\rho_1 = (1+\phi\theta)(\phi+\theta)/(1+2\phi\theta+\theta^2)$, $\rho_k = \phi^{k-1}\rho_1$; S&P 500 GARCH-N: $\hat\omega = 0{,}0254$, $\hat\alpha = 0{,}1206$, $\hat\beta = 0{,}8599$

- **Cerințe**

  1.  Înlocuiți $\phi = \alpha+\beta$, $\theta = -\beta$ și deduceți $\rho_1(\varepsilon^2)$.

  2.  Calculați $\rho_1$, $\rho_{10}$ și reprezentați ACF teoretică față de ACF de selecție a lui $(r_t - \bar r)^2$, lagurile 1--50.

  3.  Simulați 200 de traiectorii GARCH-N de lungimea eșantionului (500 de zile de burn-in eliminate; inovații Normale din `np.random.default_rng(44)`, dispersie inițială $\omega/(1-\alpha-\beta)$) și adăugați anvelopa de 95% a ACF de selecție.

- **Raportați**: formula, $\rho_1$, $\rho_{10}$, graficul; interpretare: este concordanța ACF a lui $r_t^2$ o verificare puternică sau slabă?

- Model: [A1](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A1)

- Notebook: secțiunea [A6](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A6)

[Exercițiul A6 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A6)

In [12]:
# Codul vostru aici


## Partea B — calcul și inferență

### B1. GARCH-t pe S&P 500 cu diagnostic [Rezolvat]

- Teste ARCH pe $r_t^2$; GARCH(1,1)-t cu erori standard robuste și clasice; $Q(10)$ pe $\hat z_t$ și $\hat z_t^2$, Lagrange Multiplier (LM)(5), testele de asimetrie.
- Interpretare:
  - ce teste de diagnostic resping modelul?
  - ce modificări sugerează?

In [13]:
# Rezolvare
r = rets['sp500']
print('Q(10) on r^2:', np.round(ljung_box(r ** 2, 10), 4), ' LM(5):', np.round(arch_lm(r - r.mean(), 5), 4))
f = FITS[('sp500', 'GARCH', 't')]
print(f.res.summary())
print('classical s.e.:', f.res.model.fit(disp='off', cov_type='classic', starting_values=f.res.params.values).std_err.round(4).to_dict())
z = f.z.dropna()
print('Q(10) z:', np.round(ljung_box(z, 10), 3), ' Q(10) z^2:', np.round(ljung_box(z ** 2, 10), 3), ' LM(5):', np.round(arch_lm(z, 5), 3))
{k: round(v, 3) for k, v in sign_bias_test(z, f.eps).items()}

Q(10) on r^2: [6132.8172    0.    ]  LM(5): [1683.8597    0.    ]
                        Constant Mean - GARCH Model Results                         
Dep. Variable:                        sp500   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -9062.47
Distribution:      Standardized Student's t   AIC:                           18134.9
Method:                  Maximum Likelihood   BIC:                           18169.0
                                              No. Observations:                 6714
Date:                      Thu, Oct 15 2026   Df Residuals:                     6713
Time:                              14:10:55   Df Model:                            1
                                 Mean Model                                 
                 coef    std err          t      P>|t|      95.0% Conf. Int.
---------------

{'sign_t': np.float64(3.564),
 'neg_size_t': np.float64(1.129),
 'pos_size_t': np.float64(-2.613),
 'joint': np.float64(40.665),
 'joint_p': np.float64(0.0)}

### B2. GJR față de GARCH pe S&P 500 [Rezolvat]

- $\hat\gamma$ cu $t$-ul robust; testul Likelihood Ratio (test) (LR) pentru $\gamma = 0$; Akaike Information Criterion (AIC) și Bayesian Information Criterion (BIC).
- Interpretare: este asimetria mare din punct de vedere economic?

In [14]:
# Rezolvare
lr_tests()
show('GJR vs GARCH, S&P 500', OUT['lr']['sp500'])

GJR vs GARCH, S&P 500
  Log-likelihood, GARCH: -9062.4733
  Log-likelihood, GJR: -8944.6470
  AIC, GARCH: 18134.9466
  AIC, GJR: 17901.2940
  BIC, GARCH: 18169.0063
  BIC, GJR: 17942.1657
  LR statistic: 235.6525
  LR p-value, chi2(1): 0.0000
  gamma: 0.2052
  Robust t of gamma: 10.0277
  alpha: 0.0000
  beta: 0.8809


### B3. GJR față de GARCH pe BET [Propus]

- Replicare: calcul/cod reproductibil, comparație cu reperul din curs și interpretare.

- **Întrebarea**: este asimetria bursei de la București la fel de puternică precum cea a S&P 500?

- **Date**: BET, 6 670 de randamente zilnice (Datele folosite); aceleași două modele și același cod ca în [B2](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B2); $\ln n = 8{,}81$

- **Cerințe**

  1.  Estimați ambele modele și completați tabelul de comparație din [B2](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B2) ($k$, $\ell$, AIC, BIC) pentru BET.

  2.  Calculați $LR$ și comparați-l cu cele patru praguri: 2 (AIC), 3,84 și 6,63 ($\chi^2_1$ la 5% și 1%) și $\ln n$ (BIC); precizați fiecare decizie.

  3.  Calculați impactul știrilor pentru BET la $\pm 3$% pornind de la dispersia de selecție, ca în [B2](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B2).

- **Raportați**: tabelul, cele patru decizii, cele două valori de impact; interpretare:

  - de ce pot LR și BIC să conducă la decizii diferite?

  - dovedește un $\hat\gamma$ mai mic că BET este mai puțin asimetric decît S&P 500?

- Model: [B2](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B2)

- Notebook: secțiunea [B3](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B3)

[Exercițiul B3 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B3)

In [15]:
# Codul vostru aici


### B4. Ipoteza α+β = 1: bootstrap și test la frontieră [Rezolvat] — după curs

- 999 de traiectorii simulate din GARCH(1,1)-t estimat pentru S&P 500, reestimate; intervale bazate pe percentile pentru $\alpha+\beta$ și timpul de înjumătățire.
- Algoritmul 2, testul Integrated GARCH (IGARCH): 999 de traiectorii simulate sub $H_0$ din IGARCH(1,1)-t estimat; p-value-urile bootstrap pentru LR și pentru $\hat\alpha+\hat\beta$.
- Interpretare: de ce un interval bazat pe percentile, trunchiat la 1, nu este un test pentru IGARCH?

In [16]:
# Rezolvare
boots = {'sp500': param_bootstrap('sp500')[0]}
nulls = {'sp500': igarch_test('sp500')}
show('Parametric bootstrap, S&P 500', OUT['boot_sp500'])
show('IGARCH test under H0, S&P 500', OUT['igarch_sp500'])

Parametric bootstrap, S&P 500
  Bootstrap paths B: 999
  Estimated persistence p: 0.9947
  Half-life (days): 130.7335
  2.5% percentile of p*: 0.9833
  97.5% percentile of p*: 1.0000
  Standard deviation of p*: 0.0047
  Mean of p*: 0.9935
  Half-life at the lower end of p: 41.0713
  Median half-life (days): 114.8431
  97.5% percentile of the half-life: inf
  Share of draws with an infinite half-life: 0.0891
  Share of draws with p* >= 0.999: 0.1301
  Robust s.e. of alpha + beta: 0.0047
IGARCH test under H0, S&P 500
  Bootstrap paths B: 999
  Persistence p = alpha + beta: 0.9947
  alpha under H0 (IGARCH-t): 0.1272
  omega under H0: 0.0142
  nu under H0: 6.0336
  LR statistic: 1.1080
  p-value, naive chi2(1): 0.2925
  p-value, mixture 1/2 chi2(0) + 1/2 chi2(1): 0.1463
  Bootstrap p-value of alpha + beta: 0.1670
  Bootstrap p-value of LR: 0.1750
  Share of re-estimates with alpha + beta >= 0.999: 0.5205
  95% quantile of LR*: 3.3595
  5% quantile of p* under H0: 0.9908


### B5. Compararea prognozelor pe S&P 500 [Rezolvat] — după curs

- 2016–2026, reestimare anuală; quasi-likelihood loss (QLIKE), Diebold–Mariano față de GARCH-t, Mincer–Zarnowitz; aproximarea $r_t^2$, cu media ignorată ($\hat\mu^2 \approx 0{,}006$ față de o dispersie medie de 1,47).
- Interpretare:
  - ce modele sînt semnificativ mai slabe?
  - de ce este $R^2$ mic?

In [17]:
# Rezolvare
forecast_comparison('sp500')
pd.DataFrame(OUT['fc_sp500']['models']).T.round(4)

,qlike,dm_diff,dm_t,dm_p,mz_a,mz_b,mz_b_se,mz_r2,mz_wald_p
GARCH-N,0.7510,-0.0060,-1.4733,0.1407,0.0250,0.9734,0.2728,0.2498,0.9951
GARCH-t,0.7570,0.0000,NaN,NaN,0.0844,0.8799,0.2501,0.2380,0.8042
GJR-t,0.7319,-0.0250,-1.2177,0.2233,0.1298,0.8352,0.2168,0.2764,0.6802
EGARCH-t,0.7454,-0.0115,-0.3584,0.7200,-0.2332,1.2381,0.3059,0.2933,0.7130
EWMA,0.8149,0.0579,3.4770,0.0005,0.1413,0.8863,0.2723,0.1718,0.6851
Hist-20,0.8888,0.1319,4.1020,0.0000,0.3997,0.6845,0.2243,0.1519,0.0277


### B5 extins. Giacomini–White și Model Confidence Set [Rezolvat]

- Giacomini–White cu fereastră mobilă de 1000 de zile, reestimată în fiecare ianuarie; Model Confidence Set 90% pe pierderile QLIKE ale prognozelor cu fereastră în expansiune ($T_R$, bootstrap staționar, bloc mediu 10, 10.000 de extrageri); S&P 500 și BET.

In [18]:
# Rezolvare
GWCOLS = {'mean_diff': 'diferența medie a pierderilor', 'uncond_t': 't necondiționat', 'gw': 'statistica GW', 'gw_p': 'p GW'}
for k in ['sp500', 'bet']:
    gw_mcs(k)
    print(k, 'față de GARCH-t')
    print(pd.DataFrame(INF[f'gw_{k}']['tests']).T.rename(columns=GWCOLS).round(3))
    print('Model Confidence Set 90%:', INF[f'gw_{k}']['mcs_in'], ' p-value-urile MCS:', INF[f'gw_{k}']['mcs_p'])

sp500 față de GARCH-t
          t necondiționat  statistica GW   p GW  diferența medie a pierderilor
GARCH-N             1.868          5.850  0.054                          0.008
GJR-t              -0.177          2.490  0.288                         -0.005
EGARCH-t            0.075          3.213  0.201                          0.003
EWMA                3.200         11.399  0.003                          0.072
Hist-20             3.873         16.412  0.000                          0.146
Model Confidence Set 90%: ['EGARCH-t', 'GARCH-N', 'GARCH-t', 'GJR-t']  p-value-urile MCS: {'Hist-20': 0.001, 'EWMA': 0.0084, 'GARCH-t': 0.3532, 'GARCH-N': 0.5856, 'EGARCH-t': 0.5856, 'GJR-t': 1.0}


bet față de GARCH-t
          t necondiționat  statistica GW   p GW  diferența medie a pierderilor
GARCH-N             0.948          0.897  0.639                          0.012
GJR-t              -1.708          3.364  0.186                         -0.022
EGARCH-t           -1.231          2.587  0.274                         -0.021
EWMA                3.252         11.132  0.004                          0.100
Hist-20             3.653         14.214  0.001                          0.147
Model Confidence Set 90%: ['EGARCH-t', 'EWMA', 'GARCH-N', 'GARCH-t', 'GJR-t']  p-value-urile MCS: {'Hist-20': 0.0263, 'GARCH-t': 0.1082, 'EWMA': 0.1082, 'GARCH-N': 0.6585, 'EGARCH-t': 0.6585, 'GJR-t': 1.0}


### B6. Distribuția inovațiilor pentru Bitcoin [Propus]

- Replicare: calcul/cod reproductibil, comparație cu reperul din curs și interpretare.

- **Întrebările**

  - ce distribuție a inovațiilor descrie Bitcoin?

  - este validă inferența QMLE Gaussian în acest caz?

- **Date**: prețurile de închidere ale Bitcoin (BTC/United States dollar (USD)) din 2014-09-17; randamente 2014-09-18 -- 2026-09-18, 4 384 de randamente pe zile calendaristice; GARCH(1,1), medie constantă

- **Cerințe**

  1.  Estimați cu inovații Normale și Student-t; comparați $\ell$ și BIC și reprezentați graficele quantile–quantile plot (QQ) ale lui $\hat z_t$ față de fiecare lege estimată.

  2.  Testați distribuția Normală ($1/\nu = 0$) față de Student-t prin LR cu legea la frontieră $\tfrac12\chi^2_0 + \tfrac12\chi^2_1$ (valoarea critică la 5%: 2,71).

  3.  Opțional: adăugați t asimetric ($\nu$, $\lambda$) și Generalised Error Distribution (GED, forma $\nu_G$); testați $\lambda = 0$ prin LR; raportați fiecare parametru de formă cu e.s. și semnalați limitele active.

- **Raportați**: un tabel cu $\ell$, BIC, persistență și parametrii de formă, graficul QQ, testele LR; interpretare:

  - cu $\hat\nu < 4$, este validă inferența QMLE Gaussian [Hall & Yao (2003)](https://doi.org/10.1111/1468-0262.00396)?

  - prin ce diferă pentru Maximum Likelihood Estimation (MLE) Student-t?

- Model: [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B1), [B2](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B2)

- Notebook: secțiunea [B6](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B6)

[Exercițiul B6 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B6)

In [19]:
# Codul vostru aici


### B7. Compararea prognozelor pe BET [Propus]

- **Întrebarea**: rămîn valabile concluziile din [B5](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B5) pentru bursa de la București?

- **Date**: BET, 2 675 de zile de prognoză, 2016--2026; aceleași șase modele, fereastră în expansiune, aproximare și cod ca în [B5](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B5)

- **Cerințe**

  1.  Calculați QLIKE mediu, $t$ Diebold–Mariano (test) (DM) față de GARCH-t și $\sum d_t$ cumulat.

  2.  Estimați regresia Mincer–Zarnowitz (regression) (MZ) pentru GARCH-t cu erori Heteroskedasticity and Autocorrelation Consistent (standard errors) (HAC); raportați $a$, $b$, e.s., $R^2$ și $p$ Wald și reprezentați graficul de calibrare.

  3.  Puneți rezultatele S&P 500 și BET alături, cu fereastra de estimare a fiecărei cifre.

- **Raportați**: două tabele și două grafice; interpretare:

  - sînt calibrate prognozele pentru BET?

  - ce ați testa în continuare?

- Model: [B5](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B5)

- Notebook: secțiunea [B7](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B7)

[Exercițiul B7 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B7)

In [20]:
# Codul vostru aici


### B8. Bootstrap pentru BET [Propus]

- Replicare: calcul/cod reproductibil, comparație cu reperul din curs și interpretare.

- **Întrebările**

  - cît de precisă este persistența BET?

  - poate dispersia ei să aibă o rădăcină unitară?

- **Date**: BET GARCH-t, $\hat p = 0{,}9907$ (timp de înjumătățire 74 de zile); cei doi algoritmi și funcțiile din [B4](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B4), $B = 999$; lungimea eșantionului BET, 500 de zile de burn-in, dispersia sa de selecție la inițializare, valori de inițializare 42/43

- **Cerințe**

  1.  Rulați algoritmul 1 și raportați intervalul bazat pe percentile pentru $p$, intervalul timpului de înjumătățire și proporția extragerilor la $p = 1$.

  2.  Comparați abaterea standard bootstrap a lui $\hat p^*$ cu e.s. robustă a lui $\hat\alpha + \hat\beta$.

  3.  Rulați algoritmul 2 și comparați p-value-urile bootstrap, din amestec și din $\chi^2_1$, la 5%.

- **Raportați**: intervalele, cele două e.s., testul și graficul cu două panouri; interpretare: ordonează intervalele separate pentru BET și S&P 500 cei doi timpi de înjumătățire?

- Model: [B4](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B4)

- Notebook: secțiunea [B8](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B8)

[Exercițiul B8 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B8)

In [21]:
# Codul vostru aici


## Partea C — întrebări deschise și critica unui răspuns AI

### C1. Critica unui răspuns AI [Propus]

- **Cererea**: „`px` conține închiderile zilnice S&P 500. Estimează un GARCH(1,1) cu inovații Student-$t$ cu `arch` și dă-mi volatilitatea anualizată de termen lung și timpul de înjumătățire al unui șoc."

- **Date**: `px = load_close(’sp500’)`, 6 715 închideri, 2000-01-03 -- 2026-09-18

- **Rezultatul răspunsului**: `ann_vol` = 66,46, `half_life` = 0,74 zile și un `DataScaleWarning`

- **Cerințe**

  1.  Rulați codul și reproduceți rezultatul.

  2.  Găsiți erorile de formulă, erorile de unitate și riscurile numerice, fiecare cu numărul liniei.

  3.  Pentru fiecare, indicați dovada care o evidențiază (un avertisment, o cifră din [A1](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A1) sau [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B1)) și linia corectată.

  4.  Rulați codul corectat și notați în `AI_ERRORS.md` linia, tipul erorii, dovada, corecția și rezultatul verificării.

- Model: [A1](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-A1), [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B1); anualizarea: [[slide: Unități și dispersia pe mai multe zile]{style="color: MainBlue"}](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=sem5p5) și slide-ul „Datele folosite"

``` {.numberLines numbers="left" numberstyle="\\tiny"}
import numpy as np
from arch import arch_model
r = np.log(px).diff().dropna()
am = arch_model(r, vol='GARCH', p=1,
                q=1, dist='t')
res = am.fit(disp='off')
om = res.params['omega']
a = res.params['alpha[1]']
b = res.params['beta[1]']
lr_var = om / (1 - a - b)
ann_vol = lr_var * 252
half_life = np.log(0.5) / np.log(b)
```

[Exercițiul C1 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-C1)

In [22]:
# Codul dumneavoastră corectat pentru C1


### C2. Efectul știrilor din SUA asupra volatilității BET [Propus]

- **Întrebările**

  - crește șocul S&P 500 de ieri dispersia BET de azi?

  - cu cît?

- **Date**: prețurile de închidere BET și S&P 500, 2000-01-07 -- 2026-09-18; se aliniază *prețurile* pe zilele comune, apoi se calculează randamentele; $t$ indexează zilele comune

- **Cerințe**

  1.  Construiți eșantionul zilelor comune și $u_{t-1}$, randamentul S&P 500 din ziua comună precedentă (Wall Street închide după București).

  2.  Verificați săptămîna cu sărbători de mai jos: pe 2 decembrie 2025, $u_{t-1}$ este randamentul S&P 500 de la 26 la 28 noiembrie (0,69%); precizați ce ședință americană lipsește și în ce randament este inclusă.

- **Raportați**: numărul zilelor comune și verificarea momentului

- Model: [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B1), [B2](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B2)

- Notebook: secțiunea [C2](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-C2)

::: center
:::

- Marcaje pline: zile de tranzacționare; pătrate: zilele păstrate după alinierea prețurilor; cifrele sînt $u_{t-1}$ pentru fiecare zi comună

- **Întrebările**: aceleași întrebări, acum ca model estimat

- **Date**: $r_t = \mu + \phi u_{t-1} + \varepsilon_t$, $\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \beta\sigma_{t-1}^2 + \delta u_{t-1}^2$, $\varepsilon_t = \sigma_t z_t$ cu $z_t$ Student-t standardizat; funcțiile din notebook `spill_negll`, `fit_spill`

- **Cerințe**

  1.  Scrieți log-verosimilitatea pas cu pas (pornire $\sigma_1^2$ = dispersia de selecție) și estimați modelul nerestricționat și modelul cu $\delta = 0$.

  2.  Testați $\delta = 0$ prin LR; deoarece $\delta \geq 0$, folosiți $\tfrac12\chi^2_0 + \tfrac12\chi^2_1$; repetați pe 2000--2012 și 2013--2026.

  3.  Calculați termenul american cumulat $s_t = \beta s_{t-1} + \delta u_{t-1}^2$ cu $s_1 = 0$, partea $s_t/\sigma_t^2$ și volatilitatea fără el, $\sqrt{\sigma_t^2 - s_t}$; reprezentați-le.

- **Raportați**: estimările pe trei eșantioane, testele LR, graficul contribuției; un folder Quantlet (README, script/notebook, sursa datelor, figură și metainfo) și un slide cu concluzia și principala limită; interpretare: legătura este cauzală sau predictivă?

[Exercițiul C2 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-C2)

In [23]:
# Codul vostru aici


### C3. GARCH bazat pe cvasi-scor pe acțiuni americane [Propus]

- **Întrebarea**: este grosimea cozilor randamentelor același lucru cu atenuarea șocurilor mari în actualizarea dispersiei [Blasques, Francq & Laurent (2023)](https://doi.org/10.1016/j.jeconom.2021.12.005)?

- **Date**: începeți cu JPMorgan (prețuri de închidere ajustate) și Cboe Volatility Index (VIX) (valori de închidere), aliniate pe zilele comune, 3 ianuarie 1995 -- 28 februarie 2019; $r_t = 100\,\Delta\ln P_t$, $X_t = \mathrm{VIX}_t^2/252$ (VIX din ziua $t$ intră în $f_{t+1}$)

- **Cerințe**

  1.  Modelul (recursia estimată): $y_t = r_t - \mu - \phi r_{t-1} = \sqrt{f_t}\,\epsilon_t$, $\epsilon_t$ Student-t standardizat cu $1/\xi$ grade de libertate; $f_{t+1} = \omega + \varpi X_t + \alpha\,\Psi\!\big(\tfrac{1+\zeta}{1-2\zeta+\zeta\epsilon_t^2}\big)\epsilon_t^2 f_t + \beta f_t$, $\Psi(x) = x\tanh(cx/2)$, $c = 1\,000$, $\zeta \in (-1, 1/2)$, $f_1$ = dispersia lui $y_t$; $\varpi$: ponderea termenului VIX $X_t$.

  2.  Parametri: $\xi$ = grosimea cozilor, $\zeta$ = atenuarea; $\zeta = 0$: GARCH(1,1)-t; $\zeta = \xi$: Beta-t-GARCH. Estimați cele trei modele prin verosimilitate maximă (MLE, Maximum Likelihood Estimation) (notebook-ul `MFM_ch5_quasi_score_driven`: `qsd_negll`, `fit_qsd`).

  3.  Testați $\zeta = 0$ și $\xi = \zeta$ prin LR față de $\chi^2_1$ ($\zeta = 0$ este interior datorită lui $\Psi$).

- **Raportați**: estimările, log-verosimilitățile, cele două teste LR

- Model: [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B1), [B2](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-B2)

- **Întrebările**

  - sînt adecvate cele trei modele?

  - rămîne răspunsul valabil dincolo de o acțiune?

- **Date**: estimările din cerința 1; PIT-ul $u_t = F(\epsilon_t)$ ([[slide: Forma ARMA, PIT și mulțimi de modele]{style="color: MainBlue"}](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=sem5x2)); $T$: lungimea eșantionului

- **Cerințe**

  1.  Reprezentați $\sqrt{f_t}$ pentru cele trei modele în jurul celui mai mare șoc și histogramele PIT.

  2.  Calculați statistica naivă $(\sum_t u_t - T/2)/\sqrt{T/12}$ și explicați de ce ignoră eroarea de estimare (corecția din [Pierce (1982)](https://doi.org/10.1214/aos/1176345788) scade dispersia sub $T/12$).

  3.  Opțional: extindeți la cele 11 acțiuni ale cursului și pînă la 18 septembrie 2026; raportați frecvențele de respingere cu un interval binomial care presupune, ipotetic, respingeri independente.

- **Raportați**: un folder Quantlet (README, script/notebook, sursa datelor, figură și metainfo) și un slide cu tabelul, testele și principala limită; interpretare: de ce nu pot 11 companii selectate, încă listate, confirma cele 89,5% din lucrare?

[Exercițiul C3 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar5_modele_garch_ro.pdf#nameddest=ex-C3)

In [24]:
# Codul dumneavoastră pentru C3
